# Module 2: Python Advanced

**EJ** · Stage 2 · Nov 9–13, 2026 · image: python-training:2.0

## Section 2.1: NumPy

### B-2.1.1

Create a 1,000,000-element array, compute its mean with NumPy and with a Python loop, time both with `%timeit`, and record the ratio (Lab 2.1 step 1).

In [ ]:
import numpy as np

# Seed so that the random values given is always the same everytime the code runs
rng = np.random.default_rng(7) 
array = rng.random(1_000_000)

def loop_mean(values):
    total = 0
    for value in values:
        total += value
    return total / len(values)

# timing the numpy and the loop mean computation
t_np = %timeit -o np.mean(array)
t_loop = %timeit -o loop_mean(array)
print(f"NumPy is about {t_loop.average / t_np.average:.0f}x faster")


217 μs ± 19.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
51.2 ms ± 1.33 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
NumPy is about 237x faster


**Why a DataFrame can have different dtype per column while a Numpy array has one dtype for everything?**

NumPycan only handle one datatype because it stores everything in one single block of memory, if an input with a different type gets mixed, everything gets upcast to one common type. Dataframe keeps each column as a separate NumPy array and links them like a table that's why it can support multiple data types. In the `sales` data, `quantity` is float64 and `customer` is object, that's only possible because each column is its own NumPy array.                                                                                        

### B-2.1.2

Build a 5x4 array of simulated daily sales and compute the total per store, the average per week, and a boolean mask of cells above the overall mean (Lab 2.1 step 2).

a seeded 5×4 sales array (5 stores × 4 weeks); total per store (axis 1), average per week (axis 0), mask of cells above the overall mean; label every aggregation with its axis

In [ ]:
rng = np.random.default_rng(42) 
sales = rng.integers(20_000, 80_000, size=(5, 4))   # 5 stores (rows) × 4 weeks (columns)
stores = ["Makati", "Mandaluyong", "Alabang", "Taguig", "Rizal"]
weeks = ["W1", "W2", "W3", "W4"]

sales_per_store = sales.sum(axis=1)
average_per_week = sales.mean(axis=0)
overall_average = sales.mean()
higher_than_average_mask = sales > overall_average

def print_summary(labels, values):
    """Prints the summary of each value with its label""" 
    for label, value in zip(labels, values, strict=True):
        print(f"  {label}: ₱{value:,.2f}")

print("Total sales per store (axis=1, across the 4 weeks):")
print_summary(stores, sales_per_store)
print("\nAverage per week (axis=0, across the 5 stores):")
print_summary(weeks, average_per_week)

print(f"\nOverall average (no axis, all {sales.size} cells): ₱{overall_average:,.2f}")
print(f"Store-weeks above the overall average: {higher_than_average_mask.sum()} of {sales.size}")
for row, col in np.argwhere(higher_than_average_mask):
    print(f"  {stores[row]} {weeks[col]}: ₱{sales[row, col]:,}")

Total sales per store (axis=1, across the 4 weeks):
  Makati: ₱197,398.00
  Mandaluyong: ₱204,493.00
  Alabang: ₱187,863.00
  Taguig: ₱260,024.00
  Rizal: ₱195,886.00

Average per week (axis=0, across the 5 stores):
  W1: ₱43,672.20
  W2: ₱51,391.20
  W3: ₱53,890.00
  W4: ₱60,179.40

Overall average (no axis, all 20 cells): ₱52,283.20
Store-weeks above the overall average: 10 of 20
  Makati W2: ₱66,437
  Makati W3: ₱59,274
  Mandaluyong W2: ₱71,515
  Mandaluyong W4: ₱61,842
  Alabang W4: ₱78,537
  Taguig W1: ₱64,145
  Taguig W2: ₱65,668
  Taguig W3: ₱63,048
  Taguig W4: ₱67,163
  Rizal W3: ₱70,384


### B-2.1.3

Demonstrate broadcasting by applying a per-store tax rate vector to the matrix (Lab 2.1 step 3).

In [ ]:
tax = np.array([1.18, 1.12, 1.04, 1.15, 1.05]) # made-up rates, one per store on the tax
tax_column = tax.reshape(5, 1)
sales_with_tax = sales * tax_column

print(f"Shape: {sales_with_tax.shape} (same as sales: {sales.shape})")
print(sales_with_tax)
print("\nTotal sales per store with tax added (axis=1, across the 4 weeks):")
print_summary(stores, sales_with_tax.sum(axis=1))

Shape: (5, 4) (same as sales: (5, 4))
[[29918.9  78395.66 69943.32 54671.76]
 [51497.6  80096.8  28174.72 69263.04]
 [33371.52 26676.   53651.52 81678.48]
 [73766.75 75518.2  72505.2  77237.45]
 [53332.65 29070.3  73903.2  49374.15]]

Total sales per store with tax added (axis=1, across the 4 weeks):
  Makati: ₱232,929.64
  Mandaluyong: ₱229,032.16
  Alabang: ₱195,377.52
  Taguig: ₱299,027.60
  Rizal: ₱205,680.30


**What *axis* means for each aggregation**

An axis defines which direction an aggregation function aggregates the data in.

axis=0 collapses the rows (downwards) so it gives one result per column and axis=1 will squash the columns (horizontally), which gives one result per row.

In the 5 rows x 4 columns array lab, sales.mean(axis=0) gave 4 values (one average per week) and sales.sum(axis=1) gave 5 (one total per store).

## Section 2.2: Pandas Basics

### B-2.2.1

Create a synthetic `sales.csv` in `data/` with at least 200 rows and deliberately planted missing values and duplicates (Lab 2.2 step 1).

Used the sales.csv that came with the course instead of generating one. It already meets this task: 240 rows, all 7 required columns, missing values in 5 columns, and exact duplicate rows. 

B-2.2.2 below shows the counts. Same approach as orders_raw.txt in Lab 1.4.

### B-2.2.2

Load it and report shape, dtypes, nulls per column, duplicate count, and numeric summary statistics (Lab 2.2 step 2).

In [ ]:
import pandas as pd

sales = pd.read_csv("../data/module2/sales.csv")

print(f"Rows and Columns: {sales.shape} | Duplicate rows: {sales.duplicated().sum()}")

quality = pd.DataFrame({
    "dtype": sales.dtypes.astype(str),
    "nulls": sales.isna().sum(),
    "null_%": (sales.isna().mean() * 100).round(1),
    "unique": sales.nunique(),
})
print("\nQuality:")
print(quality)

print("\nSummary Statistics:")
print(sales.describe())


Rows and Columns: (240, 7) | Duplicate rows: 12

Quality:
              dtype  nulls  null_%  unique
order_id     object      0     0.0     228
customer     object      8     3.3      20
product      object      0     0.0      10
category     object      7     2.9       3
quantity    float64      6     2.5      12
unit_price  float64      5     2.1      31
order_date   object      3     1.2     134

Summary Statistics:
         quantity  unit_price
count  234.000000  235.000000
mean     6.628205   28.315617
std      3.379218  111.411715
min      1.000000    1.080000
25%      4.000000    4.500000
50%      7.000000   14.000000
75%      9.000000   25.000000
max     12.000000  999.990000


- **Missing Values:** Five columns have missing values: `customer` (8), `category` (7), `quantity` (6), `unit_price` (5), and `order_date` (3). The `order_id` and `product` columns have no missing values.
- **Duplicates:** There are 12 duplicate rows out of 240 records.
- **Summary Statistics:** The `unit_price` column has some very high values. Most prices (75%) are 25 or below, but the highest price is 999.99 (might be placeholder or a typo). The mean (28) is higher than the median (14) because a few prices are much higher than the rest. The standard deviation (111) also shows that prices vary widely.

### B-2.2.3

Select rows with `loc`, `iloc`, and a boolean filter, one example of each (Lab 2.2 step 3).

In [ ]:
first_four_rows_loc = sales.loc[0:3, ["customer", "unit_price"]] # will include index 3, return 4 rows (index 0-3)
first_three_rows_iloc = sales.iloc[0:3, [1, 5]] # will exclude index 3, returns 3 rows (index 0-2)

print("loc[0:3] — by label, end INCLUDED (4 rows):")
print(first_four_rows_loc)

print("\niloc[0:3] — by position, end EXCLUDED (3 rows):")
print(first_three_rows_iloc)

revenue_above_500 = sales[(sales['unit_price'] * sales['quantity']) > 500]

print(f"\nOrders with revenue above 500 ({len(revenue_above_500)} rows):")
print(revenue_above_500)

loc[0:3] — by label, end INCLUDED (4 rows):
       customer  unit_price
0  Fernhill Ltd       12.60
1    Ivy Studio       12.60
2           NaN        6.50
3  Summit Goods        4.95

iloc[0:3] — by position, end EXCLUDED (3 rows):
       customer  unit_price
0  Fernhill Ltd        12.6
1    Ivy Studio        12.6
2           NaN         6.5

Orders with revenue above 500 (3 rows):
    order_id          customer    product   category  quantity  unit_price  \
50   S-00051     Alder Trading    Stapler     Office       6.0      999.99   
122  S-00121  Northwind Retail      Mouse       Tech      10.0      999.99   
209  S-00201  Northwind Retail  Chair Mat  Furniture       3.0      999.99   

     order_date  
50   2026-06-04  
122  2026-04-03  
209  2026-04-21  


### B-2.3.1

Remove duplicates and decide and document the treatment for each column's missing values (Lab 2.3 step 1).

In [ ]:
# Removing the 12 duplicates

clean = sales.drop_duplicates()
print(f"After dropping duplicates: {len(clean)} rows (was {len(sales)})")

After dropping duplicates: 228 rows (was 240)


In [ ]:
# Removing the outlier 999.99 unit_prices 

cleaned_outlier_prices = clean[clean["unit_price"] != 999.99]
print(f"After cleaning outlier unit_prices: {len(cleaned_outlier_prices)} rows (was {len(clean)})")

# Dropping missing quantity or unit price
cleaned_quantity_unit_price = cleaned_outlier_prices.dropna(subset=['quantity', 'unit_price']).copy()
print(f"After dropping mssing quantity or unit price: {len(cleaned_quantity_unit_price)} rows (was {len(cleaned_outlier_prices)})")

After cleaning outlier unit_prices: 225 rows (was 228)
After dropping mssing quantity or unit price: 214 rows (was 225)


In [ ]:
cleaned_quantity_unit_price["quantity"] = cleaned_quantity_unit_price["quantity"].astype(int)
print(f"Quantity dtype after dropping missing rows: {cleaned_quantity_unit_price['quantity'].dtype}")

Quantity dtype after dropping missing rows: int64


**Why is quantity float64 before cleaning?**

Since every column in a dataframe is just a Numpy array which can hold one datatype, a NaN or Not a Number value (which is a float) forces the whole column to float. Numpy's interger type has no way to store a missing value, so pandas uses float. After dropping the missing rows, `astype(int)` turns it back.

In [ ]:
# Replacing missing Customer values with "Unknown"

missing_before = cleaned_quantity_unit_price["customer"].isna().sum()
cleaned_quantity_unit_price["customer"] = cleaned_quantity_unit_price["customer"].fillna("Unknown")
print(f"Filled {missing_before} missing customers with 'Unknown' | missing now: {cleaned_quantity_unit_price['customer'].isna().sum()}")

Filled 8 missing customers with 'Unknown' | missing now: 0


In [ ]:
# Filling the missing categories of the products based on rows that already have a category

df = cleaned_quantity_unit_price # for easier naming below
print(df.groupby("product")["category"].nunique()) # Each product maps to 1 category

# Dictionary of product:category from valid rows
lookup = df.dropna(subset=["category"]).set_index("product")["category"].to_dict()

missing_before = df["category"].isna().sum()                        
df["category"] = df["category"].fillna(df["product"].map(lookup))   

print(f"Filled {missing_before} missing categories | missing left: {df['category'].isna().sum()}")

product
Chair Mat    1
Desk Lamp    1
Keyboard     1
Marker       1
Mouse        1
Notebook     1
Pen          1
Stapler      1
USB Cable    1
Webcam       1
Name: category, dtype: int64
Filled 7 missing categories | missing left: 0


In [ ]:
## Checking remaining missing values per column (I will be keeping the missing dates as is)

print(df.isna().sum())

order_id      0
customer      0
product       0
category      0
quantity      0
unit_price    0
order_date    3
dtype: int64


### B-2.3.2

Add the `revenue` column and a `size_band` column based on revenue (Lab 2.3 step 2).

In [ ]:
df["revenue"] = df["quantity"] * df["unit_price"]
print("Revenue summary (after cleaning):")
print(df["revenue"].describe().round(2))

Revenue summary (after cleaning):
count    214.00
mean     103.08
std      103.11
min        2.16
25%       22.12
50%       59.25
75%      160.00
max      480.00
Name: revenue, dtype: float64


In [ ]:
# size_band from revenue quartiles, rounded: Small ≤ 25, Medium 25–150, Large > 150

df["size_band"] = pd.cut(
    df["revenue"],
    bins=[0, 25, 150, np.inf],
    labels=["Small", "Medium", "Large"]
)

print(f"\nOrders per size band ({len(df)} rows):")
print(df["size_band"].value_counts().sort_index())


Orders per size band (214 rows):
size_band
Small     68
Medium    88
Large     58
Name: count, dtype: int64


In [ ]:
print("\nSample rows with revenue and size_band:")
print(df[["order_id", "quantity", "unit_price", "revenue", "size_band"]].head())


Sample rows with revenue and size_band:
  order_id  quantity  unit_price  revenue size_band
0  S-00001        12       12.60   151.20     Large
1  S-00002         3       12.60    37.80    Medium
2  S-00003        10        6.50    65.00    Medium
3  S-00004         9        4.95    44.55    Medium
4  S-00005         4        4.05    16.20     Small


### B-2.3.3

Produce revenue by category, top 5 customers by revenue, and average order value by month, keeping dates as plain strings (Lab 2.3 step 3).

In [ ]:
revenue_by_category = (df.groupby("category")["revenue"].sum().sort_values(ascending=False))

print("Revenue by category:")
print(revenue_by_category.round(2))

Revenue by category:
category
Tech         14193.15
Furniture     5834.40
Office        2030.99
Name: revenue, dtype: float64


In [ ]:
named_customer = df[df["customer"] != "Unknown"]

top_5_customers = (named_customer.groupby("customer")["revenue"].sum().nlargest(5))

print("\nTop 5 customers by revenue (excluding 'Unknown'):")
print(top_5_customers.round(2))


Top 5 customers by revenue (excluding 'Unknown'):
customer
Granite Partners    1754.50
Riverside Mart      1627.80
Juniper Labs        1568.50
Pinecrest Ltd       1460.05
Quartz Systems      1432.25
Name: revenue, dtype: float64


In [ ]:
df["month"] = df["order_date"].str[:7] # extracts the yyyy-mm

avg_order_by_month = df.groupby("month").agg(revenue=("revenue", "sum"), orders=("order_id", "nunique"),) # drops the missing dates

avg_order_by_month["aov"] = (avg_order_by_month["revenue"] / avg_order_by_month["orders"]).round(2)

print("\nAverage order value by month:")
print(avg_order_by_month)


Average order value by month:
         revenue  orders     aov
month                           
2026-01  4927.47      43  114.59
2026-02  2300.46      32   71.89
2026-03  3944.40      36  109.57
2026-04  3365.87      36   93.50
2026-05  3394.75      34   99.85
2026-06  3395.59      30  113.19


### B-2.3.4

Add three `assert` statements about the cleaned data (Lab 2.3 step 4).

In [ ]:
# Negative revenue
assert (df["revenue"] >= 0).all(), f"Negative revenue in {(df['revenue'] < 0).sum()} rows"

# Duplicate order_id
assert df["order_id"].is_unique, f"{df['order_id'].duplicated().sum()} duplicate order_ids"

# Length of raw vs cleaned
assert len(df) <= len(sales), "Cleaning added rows?"

print("All checks passed")

All checks passed


### B-2.3.5

Export a cleaned CSV and a summary CSV to `data/` (Lab 2.3 step 5).

In [ ]:
df.to_csv("../data/module2/cleaned_csv.csv", index=False)

summary = revenue_by_category.reset_index() # reset the index from aggregation earlier
summary.to_csv("../data/module2/revenue_by_category.csv", index=False)

print(pd.read_csv("../data/module2/revenue_by_category.csv"))

    category   revenue
0       Tech  14193.15
1  Furniture   5834.40
2     Office   2030.99


**When a per-row apply is still justified even though it’s slower than vectorized code?**

A per-row apply() is fine when your dataset is small enough that speed doesn't matter, or when vectorizing messy logic would ruin readability (or when there's no vectorized version at all). None of the lab needed apply since revenue, size_band, and month were all vectorized.

### Cleaning decisions

| Step | Rows affected | Decision | Why |
|------|---------------|----------|-----|
| Duplicates | 12 dropped (240 → 228) | Drop | They're exact copies of other rows, so nothing is lost |
| `unit_price` = 999.99 | 3 dropped (228 → 225) | Drop | The same price appears on three unrelated products and sits about 40× above the 75% mark of 25, so it's a placeholder, not a real price |
| Missing `quantity` / `unit_price` | 11 dropped (225 → 214) | Drop the row | Revenue needs both, and there's no safe value to fill in |
| `quantity` dtype | all rows | Cast back to int | It was float only because of NaN; once those rows were gone, int is the correct type |
| Missing `customer` | 8 filled | Fill with "Unknown" | Keeps the order's revenue; "Unknown" reads as a placeholder, not a real name |
| Missing `category` | 7 filled | Fill from `product` | Every product maps to exactly one category (checked with `nunique()`), so the lookup is safe; ffill was rejected because row order has nothing to do with category |
| Missing `order_date` | 3 kept as missing | Leave as is | Dates stay as strings in Module 2; a fake value would create a bogus month. These rows drop out of the monthly table only |
| Top 5 customers | — | Exclude "Unknown" | It groups 8 different missing buyers, not one real customer |

## Diagrams

### 2-A NumPy to pandas concept map 



```mermaid
flowchart LR
    A["ndarray<br/>one dtype, one memory block<br/>shape, axis, broadcasting"] --> B["Series<br/>one column = 1 ndarray + an index"]
    B --> C["DataFrame<br/>many Series sharing one index<br/>one dtype per column"]
    A -. "vectorized math, ~192x faster than loops" .-> C
    C --> D["Inspect<br/>shape, dtypes, isna, duplicated, describe"]
    C --> E["Select<br/>loc by label, iloc by position, boolean masks"]
    C --> F["Transform<br/>dropna, fillna, astype, pd.cut, map"]
    C --> G["Aggregate<br/>groupby → sum / nunique / nlargest"]

### 2-B Transformation pipeline (Lab 2.3)

```mermaid
flowchart LR
    A["sales.csv<br/>240 rows"] --> B["Drop duplicates<br/>228 rows"]
    B --> C["Drop unit_price 999.99<br/>225 rows"]
    C --> D["Drop missing quantity / unit_price<br/>214 rows"]
    D --> E["quantity → int<br/>customer → 'Unknown' (8)<br/>category from product (7)"]
    E --> F["Add revenue, size_band, month"]
    F --> G{"3 asserts pass?"}
    G -->|yes| H["cleaned_sales.csv<br/>214 rows"]
    F --> I["groupby summaries<br/>category · top 5 customers · monthly AOV"]
    I --> J["revenue_by_category.csv"]
```

## What Broke

A debugging log with at least two real errors

## What broke and how I fixed it

| # | Where | Error | Why it happened | Fix |
|---|-------|-------|-----------------|-----|
| 1 | B-2.1.3 | `ValueError: operands could not be broadcast together with shapes (5,4) (5,)` | The 5 tax rates had shape (5,), so NumPy lined them up with the 4 week columns instead of the 5 store rows | Reshaped the rates to a (5, 1) column with `tax.reshape(5, 1)`, so each store's rate stretches across its 4 weeks |
| 2 | B-2.3.1 | `SettingWithCopyWarning` when casting quantity to int | The DataFrame came from a filter and `dropna`, so pandas couldn't tell if it was a copy or a view of the original | Chained `.copy()` right after filtering so pandas treats it as a new DataFrame. |

**Lesson:** NumPy tries to match 1D arrays to columns first, so reshape them to (n, 1) when you want to go down the rows. Also, throw `.copy()` onto filtered DataFrames right away so pandas stops yelling at you with SettingWithCopyWarning later.

## Reflection

A reflection of 3 to 5 sentences

The axis direction was a bit confusing at first, so I came up with a mind trick using a Cartesian plane. I picture axis=0 at the center going down, and axis=1 going right. But deciding how to handle missing values was the hardest part, especially for the missing category. I initially wanted to just label them "unknown," but since there were only a few and since each product belongs to exactly one category, I could fill them in from the rows that already had one. Then, since shape mismatch cost me a lot of time, my main takeaway is to always check .shape first before aligning arrays.